In [1]:
%load_ext autoreload
%autoreload 2


In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate(
    [
        ("system", "Você é um professor de {subject}. Responda em português, em até 3 frases."),
        ("human", "{question}"),
    ]
)

prompt_value = prompt.invoke({"subject": "IA", "question": "O que é um vector store?"})
prompt_value.to_messages()

[SystemMessage(content='Você é um professor de IA. Responda em português, em até 3 frases.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='O que é um vector store?', additional_kwargs={}, response_metadata={})]

In [ ]:
from langchain_core.output_parsers import StrOutputParser

from agentic_rag.llm import get_chat_model

llm = get_chat_model()
chain = prompt | llm | StrOutputParser()

answer = chain.invoke({"subject": "AI", "question": "O que é um vector store?"})
print(type(chain))
print(answer)

<class 'langchain_core.runnables.base.RunnableSequence'>
content='Um **vector store** é um sistema que armazena vetores — representações numéricas de textos, imagens ou outros dados — e permite buscar itens semelhantes por significado. Em aplicações de IA, ele é usado, por exemplo, para encontrar documentos relevantes para uma pergunta e fornecer esse contexto a um modelo.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 99, 'prompt_tokens': 36, 'total_tokens': 135, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 28, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-Ebrh0BbRPQ6CpDsJd5FTD3IeBasWv', 'service_tier': 'default', 'finish_reason': 'stop', 'logp

In [8]:
for chunk in chain.stream({"subject": "AI", "question": "Explique o que é LCEL"}):
    print(chunk, end="", flush=True)

LCEL (LangChain Expression Language) é uma forma declarativa de montar fluxos de trabalho no LangChain, conectando componentes como modelos, prompts e parsers. Você pode combiná-los com o operador `|`, formando cadeias reutilizáveis que oferecem recursos como execução assíncrona, streaming e processamento em lote.

In [9]:
questions = [
    {"subject": "IA", "question": "O que é embedding?"},
    {"subject": "IA", "question": "O que é chunking?"},
    {"subject": "IA", "question": "O que é um retriever?"},
]

answers = chain.batch(questions, config={"max_concurrency": 2})
for question, answer in zip(questions, answers, strict=True):
    print(f"> {question['question']}\n{answer}\n")


> O que é embedding?
Embedding é uma representação de dados — como palavras, imagens ou sons — em uma lista de números chamada vetor. Esses números codificam características e relações, permitindo que sistemas de IA comparem itens semelhantes: por exemplo, palavras com significados próximos tendem a ter embeddings próximos.

> O que é chunking?
Chunking é dividir um conteúdo grande em partes menores, chamadas *chunks*, para facilitar seu processamento. Em IA, isso é comum ao preparar documentos para busca e recuperação de informações; idealmente, os trechos mantêm contexto suficiente para fazer sentido.

> O que é um retriever?
Um **retriever** é um componente de IA que busca e seleciona informações relevantes em uma coleção de documentos, com base em uma pergunta ou consulta. Em sistemas de RAG, ele encontra o contexto que um modelo de linguagem usa para elaborar a resposta.



In [11]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

def to_upper(text: str) -> str:
    return text.upper()

chain_with_steps = (
    {"subject": RunnableLambda(lambda _: "culinária"), "question": RunnablePassthrough()}
    | prompt
    |llm
    | StrOutputParser()
    | RunnableLambda(to_upper)
)

chain_with_steps.invoke("Como fazer arroz soltinho?")

'LAVE 1 XÍCARA DE ARROZ ATÉ A ÁGUA SAIR QUASE TRANSPARENTE E ESCORRA BEM. REFOGUE COM UM FIO DE ÓLEO E ALHO; JUNTE 2 XÍCARAS DE ÁGUA FERVENTE E SAL, TAMPE PARCIALMENTE E COZINHE EM FOGO BAIXO, SEM MEXER, ATÉ A ÁGUA SECAR. DESLIGUE, TAMPE POR 5 MINUTOS E SOLTE OS GRÃOS COM UM GARFO.'